# LightGBM prepayment model (v1)

For each loan and month, the model predicts the **probability the loan pays off that month**. That probability is the
loan's expected SMM. Averaging it over loans (weighted by balance) and annualizing gives a predicted CPR:
`CPR = 1 − (1 − SMM)^12`.

Trained on 2000–2018, early-stopped on 2019–2021, scored on 2022 onward (see notebook 03).

In [1]:
from pathlib import Path

import numpy as np
import polars as pl
import lightgbm as lgb
import matplotlib.pyplot as plt
from sklearn.metrics import log_loss, roc_auc_score

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
TABLE = ROOT / "data" / "combined" / "model_table.parquet"  # built by notebook 03
MODEL_PATH = ROOT / "data" / "models" / "lgbm_v1.txt"

# No rate levels (mkt_rate, orig_rate): with them the model learns "7% rates = 2000-01" instead of the refi effect,
# and predicted 16 CPR on the test years vs. 8 actual. Incentive alone carries the rate information.
FEATURES = ["incentive", "age", "orig_upb", "fico", "ltv", "cltv", "dti",
            "num_borrowers", "num_units", "month",
            "first_time_buyer", "loan_purpose", "property_type", "occupancy", "channel", "state", "agency"]

COLOR = {"actual": "#0b0b0b", "predicted": "#2a78d6"}
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False,
                     "axes.grid": True, "grid.color": "#e6e5e0", "axes.axisbelow": True, "legend.frameon": False})

## 1. Load the data

In [2]:
df = pl.read_parquet(TABLE)
data = {s: df.filter(pl.col("split") == s) for s in ["train", "valid", "test"]}
X = {s: d.select(FEATURES).to_pandas() for s, d in data.items()}  # LightGBM reads pandas category columns as categorical
y = {s: d["prepay"].to_numpy() for s, d in data.items()}
{s: f"{len(v):,} rows" for s, v in y.items()}

{'train': '20,932,025 rows',
 'valid': '6,016,086 rows',
 'test': '13,089,483 rows'}

## 2. Train

Standard settings, nothing tuned yet. Training stops when the validation score hasn't improved for 50 rounds.

In [3]:
params = dict(
    objective="binary",       # payoff this month: yes / no
    learning_rate=0.1,
    num_leaves=63,            # tree size
    min_data_in_leaf=2000,    # each leaf needs plenty of loan-months, which avoids noisy splits
    feature_fraction=0.8,     # each tree sees 80% of the features
    bagging_fraction=0.5, bagging_freq=1,  # and half the rows, which is faster and less overfit
    verbose=-1,
)
train_set = lgb.Dataset(X["train"], y["train"])
valid_set = lgb.Dataset(X["valid"], y["valid"], reference=train_set)

model = lgb.train(params, train_set, num_boost_round=1000, valid_sets=[valid_set],
                  callbacks=[lgb.early_stopping(50), lgb.log_evaluation(100)])

MODEL_PATH.parent.mkdir(parents=True, exist_ok=True)
model.save_model(MODEL_PATH)
print(f"best round: {model.best_iteration}")

Training until validation scores don't improve for 50 rounds


[100]	valid_0's binary_logloss: 0.0996382


Early stopping, best iteration is:
[80]	valid_0's binary_logloss: 0.0996126
best round: 80


## 3. Scores

- **Log loss** (lower is better) measures how good the probabilities are. It is compared with a naive model that predicts
  the training-years average payoff rate for every loan.
- **AUC** (0.5 = random, 1 = perfect) measures how well the model ranks fast loans above slow ones.
- **CPR** compares the predicted and actual speed of the whole split.

In [4]:
def cpr(smm):
    return 100 * (1 - (1 - smm) ** 12)

for s in data:
    data[s] = data[s].with_columns(p=model.predict(X[s], num_iteration=model.best_iteration))

base_rate = y["train"].mean()
rows = []
for s, d in data.items():
    p, w = d["p"].to_numpy(), d["exposure"].to_numpy()
    rows.append(dict(
        split=s,
        log_loss=log_loss(y[s], p),
        naive_log_loss=log_loss(y[s], np.full(len(p), base_rate)),
        auc=roc_auc_score(y[s], p),
        actual_cpr=cpr((d["prepay"].to_numpy() * w).sum() / w.sum()),
        predicted_cpr=cpr((p * w).sum() / w.sum()),
    ))
pl.DataFrame(rows)

split,log_loss,naive_log_loss,auc,actual_cpr,predicted_cpr
str,f64,f64,f64,f64,f64
"""train""",0.074862,0.080853,0.733313,17.750755,17.860212
"""valid""",0.099613,0.10512,0.697868,26.046336,20.330801
"""test""",0.040615,0.044955,0.684025,8.371516,11.409838


## 4. Predicted vs. actual CPR

The main check for a prepayment model: does the predicted speed follow the actual speed over time,
and across incentive and loan age? Both are balance-weighted, as in notebooks 01 and 02.

In [5]:
all_rows = pl.concat(data.values()).with_columns(
    actual=pl.col("prepay") * pl.col("exposure"), predicted=pl.col("p") * pl.col("exposure"))

def cpr_by(frame, by, min_rows=2_000):
    return (frame.group_by(by)
            .agg(pl.col("actual", "predicted").sum() / pl.col("exposure").sum(), n=pl.len())
            .filter(pl.col("n") >= min_rows)
            .with_columns(cpr(pl.col("actual")).alias("actual"), cpr(pl.col("predicted")).alias("predicted"))
            .sort(by))

ts = cpr_by(all_rows, "t", min_rows=500).with_columns(date=pl.date(pl.col("t") // 12, pl.col("t") % 12 + 1, 1))
fig, ax = plt.subplots(figsize=(13, 4))
for k in ["actual", "predicted"]:
    ax.plot(ts["date"], ts[k], color=COLOR[k], lw=1.4, label=k.title())
for start, end, label in [(2019, 2022, "valid"), (2022, 2027, "test")]:
    ax.axvspan(np.datetime64(f"{start}-01-01"), np.datetime64(f"{end}-01-01"), color="#e6e5e0", alpha=0.5, lw=0)
    ax.text(np.datetime64(f"{start}-03-01"), ax.get_ylim()[1] * 0.92, label, color="#52514e")
ax.set(title="Monthly CPR: actual vs. predicted (shaded = years the model did not train on)", ylabel="CPR (%)")
ax.legend(loc="upper right")
fig.tight_layout()

In [6]:
test = all_rows.filter(pl.col("split") == "test").with_columns(
    inc_b=(pl.col("incentive") * 4).round() / 4,  # 0.25 pp steps
    age_b=pl.col("age").clip(1, 240) // 12 + 1,   # loan age in years
)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for ax, (col, label) in zip(axes, [("inc_b", "Refi incentive (pp)"), ("age_b", "Loan age (years)")]):
    d = cpr_by(test, col)
    for k in ["actual", "predicted"]:
        ax.plot(d[col], d[k], marker="o", color=COLOR[k], label=k.title())
    ax.set(title=f"Test years: CPR by {label.lower()}", xlabel=label, ylabel="CPR (%)", ylim=(0, None))
axes[0].legend()
fig.tight_layout()

In [7]:
for col in ["agency", "loan_purpose", "occupancy"]:
    display(cpr_by(test, col).select(col, "actual", "predicted", "n"))

agency,actual,predicted,n
cat,f64,f64,u32
"""fannie""",8.292028,11.202131,6510302
"""freddie""",8.460847,11.642931,6579181


loan_purpose,actual,predicted,n
cat,f64,f64,u32
"""C""",8.881194,9.767955,2148698
"""P""",8.683088,12.246252,8759853
"""R""",6.580107,9.04242,2180876


occupancy,actual,predicted,n
cat,f64,f64,u32
"""I""",9.745544,10.201009,915672
"""P""",8.287691,11.455897,11702406
"""S""",8.515126,12.068086,471405


## 5. What the model relies on

**Gain** = how much each feature improved the model, summed over all trees (share of the total).

In [8]:
gain = model.feature_importance("gain", iteration=model.best_iteration)
pl.DataFrame({"feature": FEATURES, "gain_pct": 100 * gain / gain.sum()}).sort("gain_pct", descending=True)

feature,gain_pct
str,f64
"""incentive""",41.639813
"""age""",17.621734
"""state""",10.661614
"""orig_upb""",7.868368
"""fico""",4.517593
…,…
"""ltv""",1.015379
"""property_type""",0.751573
"""first_time_buyer""",0.495025


## 6. Takeaways

- **The model beats the naive baseline in every split.** Test log loss is 0.0406 vs. 0.0450 for the naive model, and test AUC is 0.68.
- **It follows the ups and downs over time,** including the 2022 slowdown and the 2024–25 rebound. It was trained only on 2000–2018.
- **It is too high overall in the test years: 11.4 vs. 8.4 CPR.** Most of the gap comes from loans with negative incentive (rate far below today's), which pay off at 1–4 CPR. The training years never had so many borrowers locked into rates 3–5 points below the market, so the model keeps them at about 6 CPR.
- **It is too low in the 2020–21 refi boom: 20 vs. 26 CPR in the validation years.** That boom was faster than anything at the same incentive in the training years.
- **Rate levels were removed on purpose.** With `mkt_rate` and `orig_rate`, the model used them as a calendar and predicted 16 CPR on the test years. Incentive alone carries the rate information.
- **Main drivers:** incentive (42% of gain), loan age (18%), state (11%) and loan size (8%). `agency` is under 1%, consistent with notebook 02.

**Next steps:**
1. **Burnout:** add how much incentive a loan has already had and ignored (e.g. months with incentive > 0.5 pp so far).
2. **Lock-in:** let the model see how deep below market a loan is, for example by capping incentive or adding the rate ratio `orig_rate / mkt_rate`.
3. **Delinquency** last month, lagged so it doesn't leak the answer.
4. **Tuning:** learning rate and tree size, once the features are settled.